<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/transformers_agents_multiagents/Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain openai python-dotenv langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.4/802.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.3/222.3 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.6/218.6 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.5.0
    Uninstalling typing_extensions-4.5.0:
      Successfully uninstalled typing_extensions-4.5.0

In [ ]:
!openssl rand -base64 32

v2D6XGdkQNAOxQVBt0t0wZJ7m7bq4CrgC0IWyS+tmKQ=


As langchain applications get more and more complex, it becomes crucial to be able to inspect what exactly is going on inside your chain or agent. The best way to do this is with [LangSmith](https://python.langchain.com/docs/get_started/quickstart#langsmith)

**Note that LangSmith is not needed, but it is helpful.**

LangServe helps developers deploy LangChain chains as a REST API. You do not need to use LangServe to use LangChain, but it is useful for deployments.

In LangChain, the simplest and most common chain contains 3 things:
- LLM/Chat Model
- Prompt Template: Provides instructions to the language model. This controls what the language model outputs so understanding how to construct prompts and different prompting strategies is crucial.
- Output parser: These translate raw response from the language model to a more workable format, making it easy to use the output downstream.

# LLM / Chat Model
There are 2 types of language models:
- LLM: takes a string as input and returns a string.
- ChatModel: takes a list of messages as input and returns a message.
A base message interface is defined by `BaseMessage` which has two required attributes:
- content: The content of the message.
- role: The entity from which the BaseMessage is coming

Objects to easily distinguish between different roles:
- HumanMessage: A BaseMessage coming from a human/user.
- AIMessage: ...coming from an AI/assistant.
- SystemMessage: ...coming from the system.
- FunctionMessage/ToolMessage: A BaseMessage containing the  output of a function or tool call.

There's a `ChatMessage` class where you can specify the role manually.

To call a ChatModel or LLM, use .invoke()

In [ ]:
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI

You can use open source models instead of OpenAI models, see https://python.langchain.com/docs/get_started/quickstart and navigate to LLM Chain(Local)

In [ ]:
import dotenv
dotenv.load_dotenv('./.env')

False

In [ ]:
llm = OpenAI()
chat_model = ChatOpenAI()

In [ ]:
from langchain.schema import HumanMessage

text = "What would be a good company name for a company that makes colorful socks"

messages = [HumanMessage(content=text)]


In [ ]:
llm.invoke(text)

'?\n\n"Rainbow Threads" or "Vibrant Socks Co."'

In [ ]:
chat_model.invoke(messages)

AIMessage(content='ColorfulStride')

LLM.invoke and ChatModel.invoke actually both support as input any of Union[str, List[BaseMessage], PromptValue]. PromptValue is an object that defines its own custom logic for returning it's inputs either as a string or as messages. LLMs have logic for coercing any of these into a string and ChatModels have logic for coercing any of these to messages.This meas that one can directly swap LLM and ChatModel for one another in most chains without breaking anything.

### Prompt Templates

Most LLM applications do not pass user input directly into an LLM. Usually they will add the user input to a larger piece of text, called a prompt template, that provides additional context on the specific task at hand.

PromptTemplates bundle up all the logic for going from user input into a fully formatted prompt.

In [ ]:
# A simple prompttemplate example
from langchain.prompts import PromptTemplate

prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}?")
prompt.format(product="colorful socks")


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
template = "You are a helpful assistant that translates {input_language} to {output_language}."
human_template = "{text}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template)
])

chat_prompt.format_messages(input_language="English",
                            output_language="French",
                            text="I love programming.")

In [ ]:
chain = chat_prompt | chat_model

In [ ]:
chain.invoke({"text": "i love you", "input_language": "English", "output_language": "French"})

AIMessage(content="je t'aime")

### Output Parsers

These convert the raw output of a language model into a format that can be used downstream

In [ ]:
# convert chat message to a string
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

chain = chat_prompt | chat_model | output_parser

In [ ]:
chain.invoke({"text": "Good morning my brother", "input_language": "English", \
              "output_language": "Igbo"})

In [ ]:
# A simple custom output parser that converts a comma separated list into a list

from langchain.schema import BaseOutputParser

class CommaSeparatedListOutputParser(BaseOutputParser):
    """Parse the output of an LLM call to a comma-separated list."""

    def parse(self, text: str):
        """Parse the output of an LLM call."""
        return text.strip().split(", ")

In [ ]:
CommaSeparatedListOutputParser().parse("hi, bye")

The above can be combined into one chain which will take input variables, pass those to a prompt template to create a prompt, pass the prompt to a language model, and then pass the output through an (optional) output parser.

In [ ]:



from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema import BaseOutputParser

In [ ]:
class CommaSeparatedListOutputParser(BaseOutputParser[List[str]]):
    """Parse the output of an LLM call to a comma-separated list."""

    def parse(self, text: str):
        """Parse the output of an LLM call."""
        return text.strip().split(", ")

In [ ]:
template = """You are a helpful assistant who generates comma separated
lists. A user will pass in a category, and you should generate 5 objects in that
category in a comma separated list. ONLY return a comma separated list,
and nothing more."""

human_template = "{text}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template),
])
chain = chat_prompt | ChatOpenAI() | CommaSeparatedListOutputParser()

chain.invoke({"text": "clever programming tips"})

['Avoid global variables',
 'use meaningful variable names',
 'break down complex problems into smaller functions',
 'comment your code for clarity',
 'and always test your code thoroughly.']

### Retrieval Chain

Inorder to properly answer some questions, we need to provide additional context to the LLM, this can be done through *retrieval*. Retrieval is useful when one has too much data to pass to the LLM directly, the retriever can be used to fetch only the most relevant pieces and pass those in.

In this tutorial, we will populate a vector store and use that as a retrieveer.

In [ ]:
# We need to load the data we want to index
!pip install beautifulsoup4

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
# we'll be asking questions on langsmith, so let's load it's docs
loader = WebBaseLoader("https://docs.smith.langchain.com/overview")

In [ ]:
docs = loader.load()

In [ ]:
# we need to index the docs into a vectorstore. we'll use an embedding model
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

Once again, we can use opensource embedding models instead of openai. check out the documentation

In [ ]:
# Lets use the above embedding model to ingest documents into a vectorstore
# lets use FAISS as a vector store
!pip install faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 44.6 MB/s eta 0:00:00


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(docs)
vector = FAISS.from_documents(documents, embeddings)

In [ ]:
# data has been indexed in a vectorstore. Let's create a retrieval chain
# chain will take an incoming question, look up relevant documents and
# pass those documents along with the original question into an llm
from langchain.chains.combine_documents import create_stuff_documents_chain
prompt = ChatPromptTemplate.from_template("""Answer the following question based\
 only on the provided context:

 <context>
 {context}
 </context>

 Question: {input}"""
 )

document_chain = create_stuff_documents_chain(chat_model, prompt)

In [ ]:
# we can pass in the documents directly
from langchain_core.documents import Document
document_chain.invoke({
    "input": "how can langsmith help with testing?",
    "context": [Document(page_content="langsmith can let you visualize test results")]
})

'Langsmith can help with testing by allowing users to visualize test results.'

In [ ]:
# instead, let the documents first come from the retriever. This way, we can use
# the retriever to dynamically select the most relevant documents and pass those in
from langchain.chains import create_retrieval_chain

retriever = vector.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)
response = retrieval_chain.invoke({
    "input": "how can langsmith help with testing?"
})

In [ ]:
response["answer"]

'LangSmith can help with testing by allowing users to run chains over data points and visualize the outputs. Users can easily pull down a dataset and run a chain over them, logging the results to a new project associated with the dataset. The results can then be reviewed and feedback can be assigned to runs. LangSmith also provides evaluators that can be specified when initiating a test run to evaluate the results. Additionally, LangSmith offers annotation queues for manual review and annotation of runs, allowing users to assess subjective qualities and validate auto-evaluated runs.'

Checkout langchain-serve.py on how to serve llm applications
## Memory
Most LLM applications have a conversational interface. An essential component of a conversation is being able to refer to information introduced earlier in the conversation. At bare minimum, a conversational system should be able to access some window of past messages directly, a more complex system will need to have a world model that it is constantly updating, which allows it to do things like maintain information about entities and thier relationships.

a memory system needs to support two basic actions: reading and writing. A chain will interact with its memory system twice in a given run:
- After receiving the initial user inputs before executing the core logic, a chain will read from its memory system and augment the user inputs
- After executing the core logic but before returning the answer, a chain will write the inputs and outputs of the current run to memory, so that they can be referred to in future runs.

The 2 core design decisions in any memory system:
- How state is stored: https://python.langchain.com/docs/modules/memory/chat_messages/
- How state is queried: A very simple memory system might just return the most recent messages each run. A slightly more complex memory system might return a succinct summary of the past K messages. An even more sophisticated system might extract entities from stored messages and only return information about entities referenced in the current run. https://python.langchain.com/docs/modules/memory/types/

##### Conversation Retrieval Chain

in order to update retrieval, we need a new chain, this chain will take in the most recent input and the conversation history and use an llm to generate a search query.

In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain.prompts import MessagesPlaceholder

# we need a prompt we can pass into an llm to generate a search query
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    ("user", "Given the above conversation, generate a search query to \
    look up inorder to get information relevant to the conversation"),
])
retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
chat_history = [
    HumanMessage(
    content="Can LangSmith help test my LLM applications?"
    ),
    AIMessage(content="Yes!")]
retriever_chain.invoke({
    "chat_history": chat_history,
    "input": "Tell me how"
})
# this willl return documents about testing in langsmith because the llm generated
# a new query, combining the chat history with the follow up question

[Document(page_content="Skip to main content\uf8ffü¶úÔ∏è\uf8ffüõ†Ô∏è LangSmith DocsPython DocsJS/TS DocsSearchGo to AppLangSmithOverviewTracingTesting & EvaluationOrganizationsHubLangSmith CookbookOverviewOn this pageLangSmith Overview and User GuideBuilding reliable LLM applications can be challenging. LangChain simplifies the initial setup, but there is still work needed to bring the performance of prompts, chains and agents up the level where they are reliable enough to be used in production.Over the past two months, we at LangChain have been building and using LangSmith with the goal of bridging this gap. This is our tactical user guide to outline effective ways to use LangSmith and maximize its benefits.On by default‚ÄãAt LangChain, all of us have LangSmith‚Äôs tracing running in the background by default. On the Python side, this is achieved by setting environment variables, which we establish whenever we launch a virtual environment or open our bash shell and leave them set. The

In [ ]:
# change to continue the conversation with these retrieved documents in mind
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user's questions based on the context below: \n\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}")
])

document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever_chain, document_chain)

In [ ]:
chat_history = [HumanMessage(content="Can langsmith help test my llm applications?"),
                AIMessage(content="Yes!")]
response = retrieval_chain.invoke({
    "chat_history": chat_history,
    "input": "Tell me how"
})

In [ ]:
response["answer"]

'.\nAI: LangSmith simplifies the initial setup and provides tools for tracing, testing, evaluating, and monitoring LLM applications. It also allows for collaborative debugging and exporting datasets.'

### Agent
In chains, each step is known ahead of time, with agents, LLMs decide what steps to take.
*Local(open source) models are not reliable enough yet for this so we'll use OpenAI models.

One of the first things to do when building an agent is to decide what tools it should have access to. In this example, we will give the agent access to two tools:
1. The retriever, this will let it easily answer questions about LangSmith.
2. A search tool. This will let it easily answer questions that require up to date information.

In [ ]:
# set up a tool for the retriever
from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for information about LangSmith. For any questions about LangSmith, \
    you must use this tool!"
)

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
# we'll use tavily as our search tool
search = TavilySearchResults()

In [ ]:
# lets create a list of the tools we want to use
tools = [retriever_tool, search]

In [ ]:
# create an agent to use the tools
!pip install langchainhub

In [ ]:
from langchain_openai import ChatOpenAI
from langchain import hub
from langchain.agents import create_openai_functions_agent
from langchain.agents import AgentExecutor

# Get the prompt to  use
prompt = hub.pull("hwchase17/openai-functions-agent")
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
agent_executor.invoke({"input": "how can langsmith help with testing?"})



> Entering new AgentExecutor chain...
LangSmith can help with testing in several ways:

1. Test Case Generation: LangSmith can generate test cases based on the specifications or requirements of the software being tested. It uses natural language processing techniques to understand the requirements and generate test cases that cover different scenarios and edge cases.

2. Test Data Generation: LangSmith can generate test data for the test cases. It can create realistic and diverse test data that covers different input combinations and boundary conditions. This helps in ensuring comprehensive test coverage.

3. Test Execution: LangSmith can execute the generated test cases automatically. It can interact with the software being tested and validate the expected behavior against the actual behavior. This helps in identifying any defects or issues in the software.

4. Test Result Analysis: LangSmith can analyze the test results and provide insights into the quality of the software. It can 

{'input': 'how can langsmith help with testing?',
 'output': "LangSmith can help with testing in several ways:\n\n1. Test Case Generation: LangSmith can generate test cases based on the specifications or requirements of the software being tested. It uses natural language processing techniques to understand the requirements and generate test cases that cover different scenarios and edge cases.\n\n2. Test Data Generation: LangSmith can generate test data for the test cases. It can create realistic and diverse test data that covers different input combinations and boundary conditions. This helps in ensuring comprehensive test coverage.\n\n3. Test Execution: LangSmith can execute the generated test cases automatically. It can interact with the software being tested and validate the expected behavior against the actual behavior. This helps in identifying any defects or issues in the software.\n\n4. Test Result Analysis: LangSmith can analyze the test results and provide insights into the qu

In [ ]:
# we can ask it about something not in the langsmith document
agent_executor.invoke({"input": "before this one"})



> Entering new AgentExecutor chain...
I'm sorry, but I'm not sure what you're referring to. Could you please provide more context or clarify your request?

> Finished chain.


{'input': 'before this one',
 'output': "I'm sorry, but I'm not sure what you're referring to. Could you please provide more context or clarify your request?"}

In [ ]:
chat_history = [HumanMessage(content="Can LangSmith help test my LLM applications?"),
                AIMessage(content="Yes!")]


In [ ]:
response = agent_executor.invoke({
    "chat_history": chat_history,
    "input": "no, i mean nairaland?"
})



> Entering new AgentExecutor chain...
I apologize for the confusion. Nairaland is a popular Nigerian online forum where users can discuss various topics. While LangSmith cannot directly help with testing your LLM applications, you can visit Nairaland and join relevant discussions or ask questions about LLM applications. The forum members may be able to provide insights, tips, and guidance based on their experiences.

> Finished chain.


In [ ]:

chat_history = []
active = True
while active:
    user_input = input("Enter your query: ")
    if not user_input:
        print("Quitting...")
        active = False
        break
    response = agent_executor.invoke({
        "chat_history": chat_history,
        "input": user_input
    })
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response["output"]))



> Entering new AgentExecutor chain...
Hello! How can I assist you today?

> Finished chain.


> Entering new AgentExecutor chain...
Yes, I am familiar with Nairaland. Nairaland is a popular Nigerian online forum where users can discuss various topics such as news, politics, entertainment, and more. It was founded by Seun Osewa in March 2005 and has since grown to become one of the largest online communities in Nigeria. Users can create threads, post comments, and engage in discussions with other members on a wide range of topics.

> Finished chain.


> Entering new AgentExecutor chain...
Nairaland was founded in March 2005.

> Finished chain.


> Entering new AgentExecutor chain...
No, Nairaland was not founded by Sam Altman. It was founded by Seun Osewa. Sam Altman is a well-known entrepreneur and investor, but he is not associated with the founding of Nairaland.

> Finished chain.


> Entering new AgentExecutor chain...
You're welcome! If you have any more questions, feel free to a

In [ ]:
chat_history

[HumanMessage(content='Hello agent'),
 AIMessage(content='Hello! How can I assist you today?'),
 HumanMessage(content='Do you know about nairaland?'),
 AIMessage(content='Yes, I am familiar with Nairaland. Nairaland is a popular Nigerian online forum where users can discuss various topics such as news, politics, entertainment, and more. It was founded by Seun Osewa in March 2005 and has since grown to become one of the largest online communities in Nigeria. Users can create threads, post comments, and engage in discussions with other members on a wide range of topics.'),
 HumanMessage(content='when was it founded?'),
 AIMessage(content='Nairaland was founded in March 2005.'),
 HumanMessage(content='and you say it was founded by sam altman, right?'),
 AIMessage(content='No, Nairaland was not founded by Sam Altman. It was founded by Seun Osewa. Sam Altman is a well-known entrepreneur and investor, but he is not associated with the founding of Nairaland.'),
 HumanMessage(content='Alright,

In [ ]:
response["output"]

Let's take a look at how to use ConversationBufferMemory in chains which is an extremely simple form of memory that just keeps a list of chat messages in a buffer and passed those into the prompt template.

In [ ]:
from langchain.memory import ConversationBufferMemory

In [ ]:
memory = ConversationBufferMemory()
memory.chat_memory.add_user_message("hi!")
memory.chat_memory.add_ai_message("what's up?")